In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import re


# -------------------------------------------------------------------
# Retailer selector configs — add more retailers here as needed
# -------------------------------------------------------------------
RETAILER_CONFIGS = {
    "amazon.com": {
        "name": "Amazon",
        "selectors": [
            (By.CSS_SELECTOR, "#corePriceDisplay_desktop_feature_div .a-offscreen"),
            (By.CSS_SELECTOR, "span.a-price span.a-offscreen"),
            (By.ID, "priceblock_ourprice"),
            (By.ID, "priceblock_dealprice"),
            (By.CSS_SELECTOR, ".a-price .a-price-whole"),
        ],
    },
    "target.com": {
        "name": "Target",
        "selectors": [
            (By.CSS_SELECTOR, "[data-test='product-price']"),
            (By.CSS_SELECTOR, "span[data-test='current-price']"),
            (By.CSS_SELECTOR, ".styles__CurrentPriceFontSize-sc-1mf18sa-0"),
        ],
    },
    "walmart.com": {
        "name": "Walmart",
        "selectors": [
            (By.CSS_SELECTOR, "[itemprop='price']"),
            (By.CSS_SELECTOR, "span.price-characteristic"),
            (By.CSS_SELECTOR, "[data-automation='buybox-price']"),
            (By.CSS_SELECTOR, ".price-group"),
        ],
    },
    "bestbuy.com": {
        "name": "Best Buy",
        "selectors": [
            (By.CSS_SELECTOR, ".priceView-customer-price span"),
            (By.CSS_SELECTOR, "[data-testid='customer-price'] span"),
        ],
    },
    "ebay.com": {
        "name": "eBay",
        "selectors": [
            (By.CSS_SELECTOR, ".x-price-primary span.ux-textspans"),
            (By.ID, "prcIsum"),
            (By.CSS_SELECTOR, ".notranslate[itemprop='price']"),
        ],
    },
}


def get_driver():
    """Create and return a configured Chrome WebDriver."""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationDetection")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    # Mask webdriver property to reduce bot detection
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    return driver


def detect_retailer(url: str) -> dict | None:
    """Match a URL to a known retailer config."""
    for domain, config in RETAILER_CONFIGS.items():
        if domain in url:
            return config
    return None


def clean_price(raw: str) -> str:
    """Extract a clean price like $29.99 from messy text."""
    match = re.search(r"\$[\d,]+(?:\.\d{2})?", raw)
    return match.group(0) if match else raw.strip()


def scrape_price(url: str) -> dict:
    """
    Scrape the price from a supported retailer URL.
    Returns a dict with retailer name, url, and price (or error).
    """
    result = {"url": url, "retailer": "Unknown", "price": None, "error": None}

    config = detect_retailer(url)
    if not config:
        result["error"] = (
            f"Retailer not supported. Supported: {', '.join(RETAILER_CONFIGS.keys())}"
        )
        return result

    result["retailer"] = config["name"]
    driver = get_driver()

    try:
        print(f"[{config['name']}] Loading page...")
        driver.get(url)
        time.sleep(3)  # Wait for JS to render

        wait = WebDriverWait(driver, 10)

        for by, selector in config["selectors"]:
            try:
                element = wait.until(EC.presence_of_element_located((by, selector)))
                raw_text = element.get_attribute("textContent") or element.text
                if raw_text.strip():
                    result["price"] = clean_price(raw_text)
                    print(f"[{config['name']}] Price found: {result['price']}")
                    return result
            except Exception:
                continue

        result["error"] = "Price element not found. Page structure may have changed."

    except Exception as e:
        result["error"] = str(e)

    finally:
        driver.quit()

    return result


def scrape_multiple(urls: list[str]) -> list[dict]:
    """Scrape prices from multiple product URLs."""
    results = []
    for url in urls:
        print(f"\nScraping: {url}")
        result = scrape_price(url)
        results.append(result)
        print(f"  → {result['retailer']}: {result.get('price') or result.get('error')}")
    return results


# -------------------------------------------------------------------
# Example usage
# -------------------------------------------------------------------
if __name__ == "__main__":
    urls = [
        "https://www.walmart.com/ip/Tylenol-Extra-Strength-Acetaminophen-Easy-to-Swallow-Caplets-200-Ct/5405420152?classType=REGULAR&adsRedirect=true"
    ]

    results = scrape_multiple(urls)

    print("\n========== RESULTS ==========")
    for r in results:
        status = r["price"] if r["price"] else f"ERROR: {r['error']}"
        print(f"{r['retailer']:12} | {status:15} | {r['url']}")


Scraping: https://www.walmart.com/ip/Tylenol-Extra-Strength-Acetaminophen-Easy-to-Swallow-Caplets-200-Ct/5405420152?classType=REGULAR&adsRedirect=true
[Walmart] Loading page...
  → Walmart: Price element not found. Page structure may have changed.

========== RESULTS ==========
Walmart      | ERROR: Price element not found. Page structure may have changed. | https://www.walmart.com/ip/Tylenol-Extra-Strength-Acetaminophen-Easy-to-Swallow-Caplets-200-Ct/5405420152?classType=REGULAR&adsRedirect=true


# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [6]:
import feedparser
feed_url = "https://www.cvs.com/shop/merch/extra-big-deals?widgetID=nlvr052j&mc=0&icid=shop-gnav-menu-ebd"
feed_url =  "https://www.walmart.com/"
feed = feedparser.parse(feed_url)
feed

{'bozo': 1,
 'entries': [],
 'feed': {'html': {'lang': 'en-US'},
  'meta': {'http-equiv': 'Content-Security-Policy',
   'content': 'child-src &#x27;self&#x27; blob:; connect-src &#x27;self&#x27; *.1worldsync.com *.accenture.com *.akamaihd.net *.babylist.com *.buywith.com *.cloudinary.com *.cnetcontent.com *.digital-cloud.medallia.com *.doubleclick.net *.flix360.com *.flix360.io *.fullstory.com *.kampyle.co *.kampyle.com *.ksckreate.net *.perimeterx.net *.purpleportal.net *.px-cdn.net *.px-cloud.net *.pxchk.net *.quantummetric.com *.richcontext.com *.salsify.com *.sspinc.io *.stylitics.com *.syndigo.cloud *.syndigo.com *.talkshop.live *.thestable.com *.wal.co *.walmart-customcards.com *.walmart.com:* *.walmart.net *.walmartimages.com *.zeekit.www.walmart.com 649d7f7fe2stg.blob.core.windows.net ads01.groovinads.com api.bazaarvoice.com api.inhome.walmart.com aroptical-scan.wal-mart.com assets-jpcust.jwpsrv.com assets.optiwise.ai azmatch.adsrvr.org b.affil.walmart.com b.affiliates.walmart.

In [10]:
feed

{'bozo': 1,
 'entries': [],
 'feed': {'html': {'lang': 'en-US'},
  'meta': {'http-equiv': 'Content-Security-Policy',
   'content': 'child-src &#x27;self&#x27; blob:; connect-src &#x27;self&#x27; *.1worldsync.com *.accenture.com *.akamaihd.net *.babylist.com *.buywith.com *.cloudinary.com *.cnetcontent.com *.digital-cloud.medallia.com *.doubleclick.net *.flix360.com *.flix360.io *.fullstory.com *.kampyle.co *.kampyle.com *.ksckreate.net *.perimeterx.net *.purpleportal.net *.px-cdn.net *.px-cloud.net *.pxchk.net *.quantummetric.com *.richcontext.com *.salsify.com *.sspinc.io *.stylitics.com *.syndigo.cloud *.syndigo.com *.talkshop.live *.thestable.com *.wal.co *.walmart-customcards.com *.walmart.com:* *.walmart.net *.walmartimages.com *.zeekit.www.walmart.com 649d7f7fe2stg.blob.core.windows.net ads01.groovinads.com api.bazaarvoice.com api.inhome.walmart.com aroptical-scan.wal-mart.com assets-jpcust.jwpsrv.com assets.optiwise.ai azmatch.adsrvr.org b.affil.walmart.com b.affiliates.walmart.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

TypeError: type 'tqdm' is not subscriptable

In [11]:
deals

[<Walmart Resold Flash Deals: Up to 58% off + free shipping w/ $35>,
 <Certified Refurb JBL Tour One M2 Noise-Canceling Headphones for $100 + free shipping>,
 <Open Box Shokz OpenRun Bone Conduction Open-Ear Headphones for $51 + free shipping>,
 <Open Box Beats Flex Wireless Earbuds for $21 + free shipping>,
 <INIU 30W USB-C Charger Block 2-Pack for $15 + free shipping w/ Prime>,
 <4K Action Camcorder for $20 + free shipping w/ first order>,
 <BaldrTherm 2.2'' Digital Solar Thermometer and Hygrometer 2-Pack for $10 + free shipping w/ Prime>,
 <Amazon Big Spring Sale Echo & Alexa Deals: Up to 50% off + free shipping w/ Prime>,
 <Amazon Basics 400VA/255W 6-Outlet UPS Battery Backup & Surge Protector for $36 + free shipping>,
 <Google Pixel 10 128GB Android Smartphone w/ Mint Mobile Unlimited Plan for $299 + free shipping>,
 <LG UltraGear 27" 4K DisplayHDR 600 240Hz IPS G-Sync Monitor for $464 + free shipping>,
 <Open Box Vankyo Leisure 470 1080p Projector for $24 + free shipping>,
 <Dell

In [ ]:
len(deals)

In [ ]:
deals[10].describe()

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [ ]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [ ]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [ ]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [ ]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

In [ ]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.scanner_agent import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
result

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")